# 05 模型訓練結果

## 1. Research Objective｜研究目標

利用收盤後可取得的歷史市場特徵，預測各股票未來五個交易日的年化實現波動率。比較跨股票共同訓練的 Ridge 與 LightGBM，以及 15 項與移除 `Turnover` 的 14 項特徵組。

本 Notebook 讀取已保存的 Train、Validation、驗證結果及最終模型。正式訓練、調參與匯出由 `05_model_training.py` 執行。所有績效圖使用模型重訓前的 Validation 預測，Test 留給 06。

## 2. Data Loading｜載入資料

使用 04 已切分且完成 Purging 的 Train 與 Validation。日期欄為預測日，Label 結束日另列。

請選用專案的 Python 虛擬環境。若該環境尚未安裝繪圖與 Notebook kernel 套件，可安裝 `matplotlib` 和 `ipykernel`。

In [ ]:
%matplotlib inline

from pathlib import Path
import json

import joblib
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
from IPython.display import display

project = Path.cwd()
if not (project / "05_model_training.py").is_file():
    raise FileNotFoundError("請從 Notebook 所在的專案資料夾執行 kernel。")

processed = project / "data" / "processed"
tables = project / "results" / "tables"
models = project / "models"
target = "future_5d_volatility"

train = pd.read_csv(processed / "train.csv", dtype={"code": str})
validation = pd.read_csv(processed / "validation.csv", dtype={"code": str})
comparison = pd.read_csv(tables / "model_comparison.csv")
trials = pd.read_csv(tables / "hyperparameter_results.csv")
predictions = pd.read_csv(tables / "validation_predictions.csv", dtype={"code": str})
stock_metrics = pd.read_csv(tables / "validation_metrics_by_stock.csv", dtype={"code": str})
importance = pd.read_csv(tables / "feature_importance.csv")
metadata = json.loads((models / "model_metadata.json").read_text(encoding="utf-8"))

selected = comparison.loc[comparison["selected"].eq(1)].squeeze(axis=0)
assert isinstance(selected, pd.Series), "比較報表必須恰有一個選定模型。"
assert (selected["model"], selected["feature_set"]) == (metadata["model"], metadata["feature_set"])
assert len(validation) * len(comparison) == len(predictions)

plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.axisbelow": True, "grid.alpha": 0.22,
    "axes.titleweight": "bold", "axes.labelcolor": "#344054",
    "xtick.color": "#344054", "ytick.color": "#344054",
    "font.family": "DejaVu Sans",
})
blue, teal, gray, red = "#285B83", "#148172", "#768596", "#BD5447"

def run_mask(frame, model, feature_set):
    return frame["model"].eq(model) & frame["feature_set"].eq(feature_set)

summary = pd.DataFrame([
    {"Split": split, "Rows": len(frame), "Stocks": frame["code"].nunique(),
     "First date": frame["date"].min(), "Last date": frame["date"].max(),
     "Last label end": frame["label_end_date"].max()}
    for split, frame in (("Train", train), ("Validation", validation))
])
display(summary)

## 3. Feature and Target Definition｜特徵與目標

A 組使用全部 15 項特徵，B 組移除 `Turnover`。股票代碼、交易日索引與 Label 起訖日用於識別及時間稽核。目標為預測日後五個交易日平方對數報酬的年化實現波動率。

In [ ]:
identifiers = {"date", "code", "session_index", "label_start_date", "label_end_date", target}
feature_a = [name for name in train.columns if name not in identifiers]
assert len(feature_a) == 15 and "Turnover" in feature_a
display(pd.DataFrame({"Feature": feature_a, "Set A": True,
                      "Set B": [name != "Turnover" for name in feature_a]}))

## 4. Baseline Models｜比較基準

歷史波動率基準直接使用當日 `Volatility_5`。平均值基準將 Train 的 Label 平均值套用到每筆 Validation 資料。在這裡直接重算兩個基準以核對已保存的分數。

In [ ]:
baseline_names = ["Historical Volatility Persistence", "Train Mean"]
actual = validation[target].to_numpy()
baseline_values = [validation["Volatility_5"].to_numpy(), np.full(len(validation), train[target].mean())]
baseline_rows = []
for name, values in zip(baseline_names, baseline_values, strict=True):
    residual = actual - values
    baseline_rows.append({"Model": name, "Validation MAE": np.abs(residual).mean(),
                          "Validation RMSE": np.sqrt(np.square(residual).mean())})

baselines = pd.DataFrame(baseline_rows)
saved_baselines = comparison.set_index("model").loc[baseline_names]
np.testing.assert_allclose(baselines[["Validation MAE", "Validation RMSE"]],
                           saved_baselines[["validation_mae", "validation_rmse"]], atol=1e-12)
display(baselines.round(6))

## 5. Ridge Regression Training｜Ridge 實驗紀錄

`StandardScaler` 和 Ridge 在同一條 Pipeline 中擬合。超參數搜尋期間，兩者均以 Train 估計。以下顯示已完成的 alpha 實驗。

In [ ]:
ridge_trials = trials.loc[trials["model"].eq("Ridge")].copy()
ridge_trials["alpha"] = ridge_trials["parameters"].map(lambda text: json.loads(text)["alpha"])
display(ridge_trials[["feature_set", "alpha", "validation_mae", "validation_rmse", "best_in_group"]]
        .sort_values(["feature_set", "alpha"]).round(6))

## 6. LightGBM Training｜LightGBM 實驗紀錄

以 Train 擬合，使用 Validation 的非負預測 MAE 執行 Early Stopping。表中顯示 A、B 各組的最佳設定及迭代次數。

In [ ]:
lightgbm_trials = trials.loc[trials["model"].eq("LightGBM")].copy()
best_lightgbm = lightgbm_trials.loc[lightgbm_trials["best_in_group"].eq(1)]
parameters = pd.DataFrame([json.loads(value) for value in best_lightgbm["parameters"]])
display(pd.concat([
    best_lightgbm[["feature_set", "best_iteration", "validation_mae", "validation_rmse"]].reset_index(drop=True),
    parameters,
], axis=1).round(6))

## 7. Hyperparameter Analysis｜Ridge alpha 搜尋

In [ ]:
fig, ax = plt.subplots(figsize=(8.3, 4.3), layout="constrained")
for name, color in (("A", blue), ("B", teal)):
    rows = ridge_trials.loc[ridge_trials["feature_set"].eq(name)].sort_values("alpha")
    ax.plot(rows["alpha"], rows["validation_mae"], marker="o", linewidth=2, color=color,
            label="A · 15 features" if name == "A" else "B · 14 features")
alphas = sorted(ridge_trials["alpha"].unique())
ax.set(xscale="log", xlabel="Ridge alpha (log scale)", ylabel="Validation MAE",
       title="Ridge regularization search")
ax.set_xticks(alphas, labels=[f"{a:g}" for a in alphas])
ax.grid(axis="y")
ax.legend(frameon=False)
plt.show()

## 8. Model Comparison｜整體模型比較

模型依 Validation MAE 選擇，RMSE 作為次要排序指標。兩個基準一併呈現，所有分數對應相同的 Validation 樣本。

In [ ]:
model_labels = {
    "Historical Volatility Persistence": "Historical persistence",
    "Train Mean": "Train mean",
}
ordered = comparison.sort_values("validation_mae").copy()
ordered["label"] = [
    model_labels.get(model, f"{model} / {group}")
    for model, group in zip(ordered["model"], ordered["feature_set"], strict=True)
]
fig, ax = plt.subplots(figsize=(8.5, 4.6), layout="constrained")
colors = [teal if pick else gray if group == "baseline" else blue
          for pick, group in zip(ordered["selected"], ordered["feature_set"], strict=True)]
bars = ax.barh(ordered["label"], ordered["validation_mae"], color=colors, height=0.58)
ax.bar_label(bars, fmt="%.5f", padding=6)
ax.set(xlim=(0, ordered["validation_mae"].max() * 1.22), xlabel="Validation MAE (lower is better)",
       title="Validation model comparison")
ax.invert_yaxis()
ax.grid(axis="x")
plt.show()
display(ordered[["label", "validation_mae", "validation_rmse", "mae_improvement", "selected"]].round(6))

## 9. Model Interpretation｜預測與特徵分析

### 實際值與預測值

散點使用選定模型在重訓前保存的 Validation 預測。對角線表示預測與實際相等。誤差表的偏差定義為預測減實際，負值表示該波動率區間平均低估。

In [ ]:
chosen = predictions.loc[run_mask(predictions, metadata["model"], metadata["feature_set"])].copy()
assert len(chosen) == len(validation)
x, y = chosen["actual_volatility"], chosen["predicted_volatility"]
upper = max(x.max(), y.max()) * 1.04

fig, ax = plt.subplots(figsize=(6.8, 6.3), layout="constrained")
ax.scatter(x, y, s=9, alpha=0.20, color=blue, edgecolors="none", rasterized=True)
ax.plot([0, upper], [0, upper], linestyle="--", color=red, linewidth=1.3, label="Perfect prediction")
ax.set(xlim=(0, upper), ylim=(0, upper), aspect="equal",
       xlabel="Actual annualized volatility", ylabel="Predicted annualized volatility",
       title=f"Actual vs. predicted · {metadata['model']} / {metadata['feature_set']}")
for axis in (ax.xaxis, ax.yaxis):
    axis.set_major_formatter(PercentFormatter(1))
ax.grid()
ax.legend(frameon=False)
plt.show()

chosen["actual_band"] = pd.cut(x, [-np.inf, 0.20, 0.50, 1.00, np.inf],
                               labels=["<20%", "20%–<50%", "50%–<100%", "≥100%"], right=False)
chosen["bias"] = y - x
chosen["absolute_error"] = chosen["bias"].abs()
display(chosen.groupby("actual_band", observed=True).agg(
    samples=("bias", "size"), average_bias=("bias", "mean"), mae=("absolute_error", "mean"),
).round(6))

### Ridge 係數與 LightGBM 重要性

Ridge 係數來自最終 Train＋Validation 重訓模型，表示其他特徵固定時，該標準化特徵增加一個標準差時的線性預測變化。LightGBM 取 A 組在 Train 擬合、由 Validation 選出的模型之 Gain 占比。兩圖可協助解讀模型，避免把係數或重要性當作因果證據。

In [ ]:
final_model = joblib.load(models / "final_model.joblib")
if metadata["model"] == "Ridge":
    ridge = final_model.named_steps["ridge"]
    coefficients = pd.Series(ridge.coef_, index=metadata["features"]).sort_values()
    fig, ax = plt.subplots(figsize=(8.5, 6.4), layout="constrained")
    ax.barh(coefficients.index, coefficients, height=0.65,
            color=np.where(coefficients >= 0, teal, red))
    ax.axvline(0, linewidth=1, color="#344054")
    ax.set(xlabel="Coefficient per feature standard deviation",
           title=f"Standardized Ridge coefficients · Set {metadata['feature_set']}")
    ax.grid(axis="x")
    plt.show()
else:
    display(pd.DataFrame({"Selected model": [metadata["model"]], "Ridge coefficients": ["Unavailable"]}))

lightgbm_a = importance.loc[
    run_mask(importance, "LightGBM", "A") & importance["stage"].eq("train_before_refit")
].sort_values("gain")
gain_share = lightgbm_a["gain"] / lightgbm_a["gain"].sum()
fig, ax = plt.subplots(figsize=(8.5, 6.1), layout="constrained")
ax.barh(lightgbm_a["feature"], gain_share, height=0.65, color=blue)
ax.set(xlabel="Share of total gain", title="LightGBM feature importance · Set A")
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.grid(axis="x")
plt.show()

### 逐股 Validation MAE

使用同一檔股票的資料比較選定模型與歷史波動率基準。圖中列出各股樣本數，便於判讀不同覆蓋率下的結果。

In [ ]:
selected_stock = stock_metrics.loc[
    run_mask(stock_metrics, metadata["model"], metadata["feature_set"]),
    ["code", "validation_samples", "validation_mae"],
]
baseline_stock = stock_metrics.loc[
    run_mask(stock_metrics, baseline_names[0], "baseline"), ["code", "validation_mae"],
]
stocks = selected_stock.merge(baseline_stock, on="code", validate="one_to_one",
                              suffixes=("_selected", "_baseline")).sort_values("validation_mae_selected")
assert len(stocks) == validation["code"].nunique()
positions = np.arange(len(stocks))
fig, ax = plt.subplots(figsize=(9.2, 7.3), layout="constrained")
ax.barh(positions - 0.17, stocks["validation_mae_selected"], height=0.30, color=teal,
        label=f"{metadata['model']} / {metadata['feature_set']}")
ax.barh(positions + 0.17, stocks["validation_mae_baseline"], height=0.30, color=gray,
        label="Historical persistence")
ax.set_yticks(positions, labels=[f"{code}   n={count}" for code, count in
                               zip(stocks["code"], stocks["validation_samples"], strict=True)])
ax.invert_yaxis()
ax.set(xlabel="Validation MAE (lower is better)", title="Validation MAE by stock")
ax.grid(axis="x")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.09), ncol=2, frameon=False)
plt.show()

## 10. Final Model and Export｜最終重訓與交接

下表讀取已保存的模型資訊。Validation 分數屬於模型選擇階段，Train＋Validation 重訓的模型留待 06 評估。Notebook 的 Run All 不會呼叫訓練或改寫既有結果。

若要明確重跑實驗，在專案虛擬環境的終端執行 `python 05_model_training.py`。06 使用 `models/final_model.joblib` 與 `models/model_metadata.json`，依保存的順序傳入特徵並套用非負預測規則。

In [ ]:
display(pd.DataFrame([{
    "Model": metadata["model"],
    "Feature set": metadata["feature_set"],
    "Features": len(metadata["features"]),
    "Parameters": json.dumps(metadata["parameters"], sort_keys=True),
    "Validation MAE": metadata["validation_mae"],
    "Validation RMSE": metadata["validation_rmse"],
    "Refit rows": metadata["refit_rows"],
    "Last refit label end": metadata["refit_label_end"],
}]).round(6))